# Calibration and ensembling

Two predictors A and B output a probability per sample for a binary outcome. We have past predictions from both, plus the actual outcomes (e.g. the predictors say "will it rain tomorrow" and we get a 0/1 for whether it actually rained or not). Both are accurate-ish but miscalibrated in different ways: A is overconfident on one region of input space, B on the opposite region.

We will:
1. Look at reliability diagrams and Expected Calibration Error (ECE).
2. Fix each with temperature scaling.
3. Ensemble the two predictors and check whether calibration improves further.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pandas as pd
from ipywidgets import interact, FloatSlider

torch.manual_seed(0)

## Data

Load predictions and outcomes from `data/calibration_data.npz` (see `data/make_calibration_data.ipynb`). The file contains, for a validation set (used to fit calibrators) and a test set (used to evaluate):

- `pA`, `pB`: per-sample predicted probabilities from predictors A and B
- `y`: binary outcome, 0 or 1 (e.g. "did it actually rain")

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def logit(p, eps=1e-7):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

d = np.load("data/calibration_data.npz")
y_val, y_te = d["y_val"], d["y_te"]
pA_val, pB_val = d["pA_val"], d["pB_val"]
pA_te,  pB_te  = d["pA_te"],  d["pB_te"]

print(f"validation: {len(y_val)} samples, positive rate = {y_val.mean():.3f}")
print(f"test:       {len(y_te)} samples, positive rate = {y_te.mean():.3f}")

color_a = "#DF752D"
color_b = "#70AD47"

### Visualizing the two predictors

For each predictor, a stacked histogram of its predicted probabilities on the test set, split by the actual outcome (e.g. rain vs no rain). A vertical line at 0.5 marks the decision boundary. A well-calibrated bin centered at 0.8 should be roughly 80% blue (y=1); a bin near 0.5 should be roughly half-half.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)

bins = np.linspace(0, 1, 11)

for ax, p_pred, name in [
    (axes[0], pA_te, "Predictor A"),
    (axes[1], pB_te, "Predictor B"),
]:
    p_neg = p_pred[y_te == 0]
    p_pos = p_pred[y_te == 1]
    acc = ((p_pred > 0.5) == y_te).mean()
    ax.hist([p_neg, p_pos], bins=bins, stacked=True,
            color=["#d62728", "#d62728"],
            edgecolor="none")
    #ax.axvline(0.5, color="k", linestyle="--", alpha=0.5)
    ax.set_xlabel("predicted P(y=1)")
    ax.set_ylabel("count")
    ax.set_xlim(0, 1)
    ax.set_title(f"{name}")#  (accuracy = {acc:.3f})")
    #ax.legend(loc="upper center", fontsize=9)
fig.set_dpi(150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)

bins = np.linspace(0, 1, 11)

for ax, p_pred, name in [
    (axes[0], pA_te, "Predictor A"),
    (axes[1], pB_te, "Predictor B"),
]:
    p_neg = p_pred[y_te == 0]
    p_pos = p_pred[y_te == 1]
    acc = ((p_pred > 0.5) == y_te).mean()
    ax.hist([p_neg, p_pos], bins=bins, stacked=True,
            color=["#d62728", "#1f77b4"], label=["no rain", "rain"],
            edgecolor="white")
    ax.axvline(0.5, color="k", linestyle="--", alpha=0.5)
    ax.set_xlabel("predicted P(y=1)")
    ax.set_ylabel("count")
    ax.set_xlim(0, 1)
    ax.set_title(f"{name} (accuracy = {acc:.3f})")
    ax.legend(loc="upper center", fontsize=9)
fig.set_dpi(150)
plt.show()

## (A) Measuring calibration

Expected Calibration Error (ECE) bins predictions by their probability, computes the gap between the average predicted probability and the empirical positive rate (e.g. fraction of those days that actually rained) in each bin, and averages those gaps weighted by bin size:

$$ \text{ECE} = \sum_{b=1}^{B} \frac{n_b}{N}\, \left|\, \bar p_b - \bar y_b \,\right| $$

where bin $b$ contains $n_b$ samples, $\bar p_b$ is their mean predicted probability and $\bar y_b$ is the fraction that turned out positive.

A reliability diagram plots, per bin, $\bar p_b$ against $\bar y_b$. Perfect calibration is the diagonal. We also show the histogram of predicted probabilities underneath, since ECE alone hides whether a predictor ever uses the extreme bins.

In [ ]:
N_BINS = 12

def bin_stats(probs, labels, n_bins=N_BINS):
    # group predictions into equal-width probability bins and report,
    # for each bin: mean predicted prob, empirical positive rate, count
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(probs, bins) - 1, 0, n_bins - 1)
    avg_p, frac_pos, counts = [], [], []
    for b in range(n_bins):
        m = idx == b
        counts.append(int(m.sum()))
        if m.sum() == 0:
            avg_p.append(np.nan); frac_pos.append(np.nan)
        else:
            avg_p.append(probs[m].mean())
            frac_pos.append(labels[m].mean())
    return bins, np.array(avg_p), np.array(frac_pos), np.array(counts)

def ece(probs, labels, n_bins=N_BINS):
    # weighted average of |mean predicted prob - empirical positive rate| per bin
    _, avg_p, frac_pos, counts = bin_stats(probs, labels, n_bins)
    w = counts / counts.sum()
    mask = counts > 0
    return float(np.sum(w[mask] * np.abs(avg_p[mask] - frac_pos[mask])))

def plot_reliability(axes, probs, labels, title, color):
    # top: reliability curve (predicted vs empirical, diagonal = perfect)
    # bottom: histogram of predicted probabilities (where the model spends its mass)
    ax_rel, ax_hist = axes
    bins, avg_p, frac_pos, counts = bin_stats(probs, labels)
    centers = 0.5 * (bins[1:] + bins[:-1])
    ax_rel.plot([0, 1], [0, 1], 'k--', alpha=0.5, label="perfect")
    m = counts > 0
    ax_rel.plot(avg_p[m], frac_pos[m], marker='o', color=color, linewidth=2)
    ax_rel.set_xlim(0, 1); ax_rel.set_ylim(0, 1)
    ax_rel.set_xlabel("predicted P(rain)")
    ax_rel.set_ylabel("empirical rain frequency")
    ax_rel.set_title(f"{title}\nECE = {ece(probs, labels):.3f}")
    ax_rel.legend(loc='upper left', fontsize=8)
    ax_hist.bar(centers, counts / counts.sum(), width=1.0 / N_BINS,
                color=color, edgecolor='white', alpha=0.85)
    ax_hist.set_xlim(0, 1)
    ax_hist.set_xlabel("predicted P(rain)")
    ax_hist.set_ylabel("fraction of data")

### Reliability of the raw predictors

If a predictor is overconfident, its curve falls below the diagonal on the right (predicted prob too high, actual positive rate lower) and above on the left. The histograms below show whether the predictor actually uses the extreme bins.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 4), constrained_layout=True,
                         gridspec_kw={'height_ratios': [3, 1]})
plot_reliability((axes[0, 0], axes[1, 0]), pA_te, y_te, "Predictor A",  color=color_a)
plot_reliability((axes[0, 1], axes[1, 1]), pB_te, y_te, "Predictor B",  color=color_b)

for i in range(2):
    axes[i, 1].set_ylabel("")
fig.set_dpi(200)
plt.show()

## (B) Temperature scaling

Take the logit $z = \log\!\big(p / (1-p)\big)$, divide by a single scalar $T > 0$, and re-apply the sigmoid:

$$ p_T = \sigma(z / T) = \frac{1}{1 + e^{-z/T}}. $$

`T > 1` softens predictions (overconfident model), `T < 1` sharpens them. `T` is a single number, so it cannot change the ranking of predictions or fix miscalibration that depends on the input.

`T` needs an objective. The standard choice is to minimize negative log-likelihood (BCE) on a held-out validation set:

$$ T^* = \arg\min_{T>0} \; -\frac{1}{N}\sum_i \big[\, y_i \log p_T(z_i) + (1-y_i)\log(1 - p_T(z_i)) \,\big]. $$

The ECE can prefer slightly different temperatures, we fit both ways below.

In [ ]:
def fit_temperature_nll(probs, labels, n_steps=20, tol=1e-7):
    # standard approach: minimize BCE (= NLL) over T > 0
    # we convert probs back to logits z = log(p/(1-p)) and apply T to those
    # parameterize T = exp(log_T) so the optimizer is unconstrained
    L = torch.tensor(logit(probs), dtype=torch.float32)
    Y = torch.tensor(labels, dtype=torch.float32)
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=200,
                            tolerance_grad=1e-9, tolerance_change=1e-12)
    bce = nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad()
        loss = bce(L / torch.exp(log_T), Y)
        loss.backward()
        return loss
    prev = float(log_T.item())
    for _ in range(n_steps):
        opt.step(closure)
        cur = float(log_T.item())
        if abs(cur - prev) < tol:
            break
        prev = cur
    return float(torch.exp(log_T).item())

def fit_temperature_ece(probs, labels, T_grid=None):
    # alternative: minimize ECE directly on the validation set.
    # ECE is not smooth in T, so we just sweep a grid.
    if T_grid is None:
        T_grid = np.linspace(0.3, 5.0, 200)
    z = logit(probs)
    probs_T = sigmoid(z[None, :] / T_grid[:, None])
    eces = np.array([ece(p, labels) for p in probs_T])
    return float(T_grid[int(np.argmin(eces))])

def apply_temperature(probs, T):
    return sigmoid(logit(probs) / T)

T_A = fit_temperature_nll(pA_val, y_val)
T_B = fit_temperature_nll(pB_val, y_val)
T_A_ece = fit_temperature_ece(pA_val, y_val)
T_B_ece = fit_temperature_ece(pB_val, y_val)

print("Fitted temperatures on the validation set:")
print(f"  NLL-fit:  T_A = {T_A:.3f}    T_B = {T_B:.3f}")
print(f"  ECE-fit:  T_A = {T_A_ece:.3f}    T_B = {T_B_ece:.3f}")
print()
print("NLL and ECE measure different things, so they prefer different temperatures.")
print("NLL is smooth and standard; ECE-targeted fitting can give lower ECE but is noisier and")
print("more prone to overfitting the validation set (ECE depends on the binning).")

pA_te_ts = apply_temperature(pA_te, T_A)
pB_te_ts = apply_temperature(pB_te, T_B)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 4), constrained_layout=True,
                         gridspec_kw={'height_ratios': [3, 1]})
plot_reliability((axes[0, 0], axes[1, 0]), pA_te_ts, y_te,
                 f"Predictor A after T-scaling (T={T_A:.2f})", color=color_a)
plot_reliability((axes[0, 1], axes[1, 1]), pB_te_ts, y_te,
                 f"Predictor B after T-scaling (T={T_B:.2f})", color=color_b)
fig.set_dpi(200)
plt.show()

### Interactive: try different temperatures

Slide `T_A` and `T_B` to see how the reliability diagrams change. The fitted values from the previous cell are a good starting point; smaller `T` sharpens predictions further, larger `T` flattens them toward 0.5.

In [ ]:
def show_ts(T_A_widget, T_B_widget):
    pA = apply_temperature(pA_te, T_A_widget)
    pB = apply_temperature(pB_te, T_B_widget)
    fig, axes = plt.subplots(2, 2, figsize=(8, 4), constrained_layout=True,
                             gridspec_kw={'height_ratios': [3, 1]})
    plot_reliability((axes[0, 0], axes[1, 0]), pA, y_te,
                     f"Predictor A (T = {T_A_widget:.2f})", color=color_a)
    plot_reliability((axes[0, 1], axes[1, 1]), pB, y_te,
                     f"Predictor B (T = {T_B_widget:.2f})", color=color_b)
    fig.set_dpi(200)
    plt.show()

interact(
    show_ts,
    T_A_widget=FloatSlider(min=0.2, max=5.0, step=0.05, value=T_A, description="T_A"),
    T_B_widget=FloatSlider(min=0.2, max=5.0, step=0.05, value=T_B, description="T_B"),
);

## (C) Ensembling

Two ways to combine A and B:

**Mean ensemble.** Just average the probabilities:

$$ p_\text{mean} = \tfrac{1}{2}\, p_A + \tfrac{1}{2}\, p_B. $$

**Confidence-weighted ensemble.** Trust each predictor proportionally to how far it is from 0.5 (its "confidence"). With weights $w_i = |p_i - 0.5|$,

$$ p_\text{conf} = \frac{w_A\, p_A + w_B\, p_B}{w_A + w_B}. $$

T-scaling each predictor first removes some over/under-confidence, so the ensemble of T-scaled predictors should be the best by ECE.

In [ ]:
def confidence_weighted(pA, pB, eps=1e-9):
    wA = np.abs(pA - 0.5)
    wB = np.abs(pB - 0.5)
    return (wA * pA + wB * pB) / (wA + wB + eps)


ensembles = {
    "Mean (raw)":         0.5 * pA_te    + 0.5 * pB_te,
    "Conf-weighted (raw)": confidence_weighted(pA_te, pB_te),
    "Mean (T-scaled)":    0.5 * pA_te_ts + 0.5 * pB_te_ts,
    "Conf-weighted (T-scaled)": confidence_weighted(pA_te_ts, pB_te_ts),
}

fig, axes = plt.subplots(4, 2, figsize=(8, 8), constrained_layout=True,
                         gridspec_kw={'height_ratios': [3, 1, 3, 1]})
fig.set_dpi(200)
positions = [
    ("Mean (raw)",               (axes[0, 0], axes[1, 0])),
    ("Conf-weighted (raw)",       (axes[0, 1], axes[1, 1])),
    ("Mean (T-scaled)",          (axes[2, 0], axes[3, 0])),
    ("Conf-weighted (T-scaled)",  (axes[2, 1], axes[3, 1])),
]
for name, axpair in positions:
    p = ensembles[name]
    title = f"{name}"
    plot_reliability(axpair, p, y_te, title, color="#2ca02c")
plt.show()

### Summary table

ECE on the test set for each option.

In [ ]:
all_models = [
    ("Predictor A (raw)",                pA_te),
    ("Predictor B (raw)",                pB_te),
    ("Predictor A (T-scaled)",           pA_te_ts),
    ("Predictor B (T-scaled)",           pB_te_ts),
    ("Mean ensemble (raw)",              ensembles["Mean (raw)"]),
    ("Conf-weighted ensemble (raw)",     ensembles["Conf-weighted (raw)"]),
    ("Mean ensemble (T-scaled)",         ensembles["Mean (T-scaled)"]),
    ("Conf-weighted ensemble (T-scaled)", ensembles["Conf-weighted (T-scaled)"]),
]
rows = [(name, ece(p, y_te)) for name, p in all_models]
pd.DataFrame(rows, columns=["model", "ECE (test)"]).round(4)